# Turtle Breakout 04 — what separates winning from losing trades of the original turtle?

Pre-registration: `discussions/Turtle_Breakout/006_claude_tb04_preregistration.md`. Code: `src/data_engineering/modeling/turtle_trade_features.py`.

Original rules (TB 01) on ~30 Binance USD-M USDT perpetuals. **Discovery**: 26 non-core perps, full history. **Confirmation**: BTC/ETH/SOL/XRP from 2020-10-15 (the TB 01 trades).
Label: $R$ = net return / (2 ATR / entry), win = $R>0$. Entry features (price, flow/CVD, funding, volatility engine, open interest and positioning, spot–perp basis, market breadth) are known before the entry bar; path features are measured during the trade and are descriptive only.

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB04_SYNTHETIC=1`.

In [ ]:
import os, sys, json, time, gc, subprocess, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, vwap_drift as vd

SYNTHETIC = os.getenv('TB04_SYNTHETIC', '0') == '1'
P = tb.Params(); FP = tf.FParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb04-')) / '04'; FP = tf.FParams(n_boot=300, n_perm=40)
    ASSETS = ['BTC', 'ETH', 'SOL', 'XRP', 'BNB', 'DOGE', 'ADA', 'LINK', 'DOT', 'LTC']
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '04'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    ASSETS = list(tf.UNIVERSE)
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_trade_features.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
FITS = vd.load_engine(ROOT / 'notebooks' / 'Volatility_Dynamics' / '18' / '18_manifest.json')
print(json.dumps(tf.registration(FP), indent=1)[:2500])

## 1. Trades and features, asset by asset

In [ ]:
T0 = time.time(); FS, STATES, COV = [], [], []
for i, a in enumerate(ASSETS):
    t1 = time.time()
    if SYNTHETIC:
        b = tb.synthetic_5m(1800, seed=60 + i, trend=0.01, start='2020-01-01'); x = tf.synthetic_flow(b, i)
        f = tb.synthetic_funding(b, 0.0001); f['rate'] *= 1 + 0.5 * np.sin(np.arange(len(f)) / 30)
        rng = np.random.default_rng(i)
        met = pd.DataFrame({'sum_open_interest': np.exp(np.cumsum(rng.normal(0, .01, len(b) // 12))) * 1e6}, index=b.index[::12][:len(b) // 12])
        met['sum_open_interest_value'] = met.sum_open_interest * 100
        for c in ('sum_toptrader_long_short_ratio', 'count_long_short_ratio', 'sum_taker_long_short_vol_ratio'): met[c] = 1 + rng.normal(0, .05, len(met))
        spot = b.close * (1 - 0.0002 * (1 + rng.normal(0, .5, len(b))))
    else:
        pu = DATA / f'{a}USDT_um_1m.parquet'
        if not pu.exists():
            COV.append({'asset': a, 'status': 'no perp file'}); print(a, 'missing'); continue
        b, x = tf.load_um(pu)
        fpth = DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        f = tb.funding_events(pd.read_parquet(fpth)) if fpth.exists() else None
        spot = tf.load_spot_close(DATA / 'basis' / 'spot' / f'{a}USDT_spot_1m.parquet')
        met = tf.load_metrics(DATA / 'basis' / 'metrics' / f'{a}USDT_metrics.parquet')
    d = tb.prepare_asset(a, b, f, P)
    start = tb.COMMON_START if a in tf.CONFIRM_ASSETS else b.index[0] + pd.Timedelta(days=30)
    T = tf.run_trades(d, P, start)
    F = tf.entry_and_path_features(a, d, x, T, f, spot, met, FITS)
    if len(F):
        F['set'] = 'confirm' if a in tf.CONFIRM_ASSETS else 'discovery'; FS.append(F)
    STATES.append(tf.daily_states(a, d))
    COV.append({'asset': a, 'status': 'ok', 'first_bar': b.index[0], 'last_bar': b.index[-1], 'trades': len(T),
                'win_rate': float(T.win.mean()) if len(T) else np.nan, 'sum_R': float(T.R.sum()) if len(T) else 0.0,
                'funding': f is not None, 'spot': spot is not None, 'metrics': met is not None,
                'metrics_first': None if met is None else met.index[0]})
    print(f"{a}: {len(T)} trades, {time.time() - t1:.0f}s")
    del b, x, d, spot, met; gc.collect()
COV = pd.DataFrame(COV); display(COV)
FALL = tf.add_market_features(pd.concat(FS, ignore_index=True), pd.concat(STATES, ignore_index=True))
Fd, Fc = FALL[FALL.set == 'discovery'].reset_index(drop=True), FALL[FALL.set == 'confirm'].reset_index(drop=True)
print('discovery', len(Fd), 'trades, win rate', round(Fd.win.mean(), 3), '| confirmation', len(Fc), 'trades, win rate', round(Fc.win.mean(), 3))
ENTRY = [c for g in tf.ENTRY_FEATURES.values() for c in g]
GROUP = {c: g for g, cs in tf.ENTRY_FEATURES.items() for c in cs}
display(FALL.groupby('set')[ENTRY].apply(lambda g: g.notna().mean()).T.round(2))

## 2. Pre-registered univariate screen (discovery, BH-FDR) and confirmation (Holm)

In [ ]:
Ud = tf.univariate(Fd, ENTRY, 'win', FP); Ud['group'] = Ud.feature.map(GROUP)
display(Ud.sort_values('p').round(4))
CONF = tf.confirm(Ud, Fc, 'win', FP)
print(f"discovery survivors (q < {FP.fdr_q}):", int((Ud.q_bh < FP.fdr_q).sum()))
display(CONF.round(4))

## 3. Pre-registered model test: ridge logistic on all entry features

In [ ]:
MOD = tf.model_test(Fd, Fc, ENTRY, 'win', FP)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in MOD.items() if k not in ('coef', 'scores_confirm', 'features')})
display(pd.Series(MOD['coef']).head(15).round(3))
ECON = tf.tercile_economics(Fc, MOD['scores_confirm']); print('Reported only: confirmation trades by model-score tercile'); display(ECON.round(3))
DECISION = {'n_discovery': len(Fd), 'n_confirm': len(Fc), 'discovery_survivors': int((Ud.q_bh < FP.fdr_q).sum()),
            'confirmed_features': CONF.loc[CONF.confirmed, 'feature'].tolist() if len(CONF) else [],
            'model_auc_confirm': MOD['auc_confirm'], 'model_lo': MOD['lo'], 'model_perm_p': MOD['perm_p'], 'model_passed': MOD['passed']}
DECISION['verdict'] = 'ENTRY_SIGNAL_CONFIRMED' if (DECISION['confirmed_features'] or MOD['passed']) else 'NOTHING_CONFIRMED'
print(json.dumps(DECISION, indent=1, default=float))

## 4. Reported: big winners (R ≥ 3), per-group models

In [ ]:
Ub = tf.univariate(Fd, ENTRY, 'big', FP, seed0=1000); Ub['group'] = Ub.feature.map(GROUP)
CONFb = tf.confirm(Ub, Fc, 'big', FP)
display(Ub.sort_values('p').head(12).round(4)); display(CONFb.round(4))
grp = []
for g, cs in tf.ENTRY_FEATURES.items():
    m = tf.model_test(Fd, Fc, cs, 'win', tf.FParams(n_boot=FP.n_boot, n_perm=50, min_feature_n=FP.min_feature_n))
    grp.append({'group': g, 'features': len(m['features']), 'auc_confirm': m['auc_confirm'], 'lo': m['lo'], 'hi': m['hi'], 'perm_p': m['perm_p']})
GRP = pd.DataFrame(grp); display(GRP.round(3))

## 5. Inside the trades (descriptive): path features and early-path win rates

In [ ]:
PT = pd.concat([tf.path_table(Fd).assign(set='discovery'), tf.path_table(Fc).assign(set='confirm')])
display(PT.round(3))
EP = pd.concat([tf.early_path_table(Fd).assign(set='discovery'), tf.early_path_table(Fc).assign(set='confirm')])
display(EP.round(3))
print('exit reasons by outcome'); display(pd.crosstab([FALL.set, FALL.reason], FALL.win))

## 6. Figures

In [ ]:
U = Ud.dropna(subset=['auc']).sort_values('auc')
cmap = dict(zip(tf.ENTRY_FEATURES, ['#1f5aa6', '#c2571a', '#3a7d44', '#8a4fa3', '#b8860b', '#2a9d8f', '#6c757d']))
fig, ax = plt.subplots(figsize=(9, 0.26 * len(U) + 1.5))
y = np.arange(len(U))
ax.errorbar(U.auc, y, xerr=[U.auc - U.lo, U.hi - U.auc], fmt='none', ecolor='0.6', lw=1)
ax.scatter(U.auc, y, c=[cmap[g] for g in U.group], s=22, zorder=3, label='discovery')
cauc = {f: tf.auc(Fc[f].to_numpy(float), Fc.win.to_numpy()) for f in U.feature}
ax.scatter([cauc[f] for f in U.feature], y, marker='x', color='k', s=18, zorder=4, label='confirmation (point)')
ax.axvline(0.5, color='k', lw=0.7); ax.set_yticks(y); ax.set_yticklabels(U.feature, fontsize=7)
ax.set_xlabel('AUC win vs loss (0.5 = no information)'); ax.legend(frameon=False, fontsize=8, loc='lower right')
ax.set_title('Entry features: discovery AUC with 95% month-cluster CI', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for s, mk in (('discovery', 'o'), ('confirm', 's')):
    e = EP[(EP.set == s)]
    for dcol, c in zip(('R_1d', 'R_3d', 'R_5d'), ('#1f5aa6', '#c2571a', '#3a7d44')):
        g = e[e.day == dcol]; axes[0].plot(range(len(g)), g.win_rate, marker=mk, color=c, ls='-' if s == 'discovery' else ':', label=f'{s} {dcol}')
axes[0].set_xticks(range(len(g))); axes[0].set_xticklabels([str(b) for b in g.bucket], rotation=30, fontsize=7)
axes[0].set_ylabel('final win rate'); axes[0].set_xlabel('R reached after N days'); axes[0].legend(fontsize=7, frameon=False); axes[0].grid(alpha=0.25)
for s, c in (('discovery', '0.6'), ('confirm', '#1f5aa6')):
    g = FALL[FALL.set == s]; axes[1].scatter(g.mae_R, g.mfe_R, s=6 if s == 'discovery' else 14, c=np.where(g.win == 1, '#3a7d44', '#b33f40'), alpha=0.5 if s == 'discovery' else 0.9)
axes[1].set_xlabel('max adverse excursion (R)'); axes[1].set_ylabel('max favourable excursion (R)'); axes[1].set_yscale('symlog'); axes[1].grid(alpha=0.25)
axes[1].set_title('green = winner, red = loser (large points: confirmation)', fontsize=9, loc='left')
plt.tight_layout(); plt.show()

## 7. Exports

In [ ]:
FALL.drop(columns=['entry_day'], errors='ignore').to_csv(OUT / '04_trade_features.csv.gz', index=False, float_format='%.6g')
Ud.to_csv(OUT / '04_univariate_discovery.csv', index=False, float_format='%.6g'); CONF.to_csv(OUT / '04_confirmation.csv', index=False, float_format='%.6g')
Ub.to_csv(OUT / '04_univariate_big.csv', index=False, float_format='%.6g'); CONFb.to_csv(OUT / '04_confirmation_big.csv', index=False, float_format='%.6g')
GRP.to_csv(OUT / '04_group_models.csv', index=False, float_format='%.6g'); ECON.to_csv(OUT / '04_model_terciles.csv', float_format='%.6g')
PT.to_csv(OUT / '04_path_features.csv', index=False, float_format='%.6g'); EP.to_csv(OUT / '04_early_path.csv', index=False, float_format='%.6g')
COV.to_csv(OUT / '04_coverage.csv', index=False)
(OUT / '04_decision.json').write_text(json.dumps({'decision': DECISION, 'model_coef': MOD['coef'], 'registration': tf.registration(FP)}, indent=1, default=float))
(OUT / '04_manifest.json').write_text(json.dumps({'module': 'Turtle_Breakout_04', 'version': tf.VERSION, 'synthetic': SYNTHETIC,
    'assets': ASSETS, 'runtime_s': round(time.time() - T0, 1)}, indent=1))
print(sorted(p.name for p in OUT.iterdir()))